In [ ]:
## Importing Libraries

#Set seed
import random
random.seed(1234)

# Manipulating & Visualizing Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(rc={'figure.figsize':(12, 8)})

# Statiscal methods
from scipy import stats

# Sampling Methods
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import NearMiss, RandomUnderSampler

# Model Selectioin
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate

# Dimensionality Reduction
from sklearn.decomposition import PCA, TruncatedSVD

# Simple ML models
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Ensemble Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Performance metrics
import sklearn.metrics as skm

In [ ]:
filled_df = pd.read_csv("/kaggle/input/newdata/data_selected.csv", index_col=None)
data_test = pd.read_csv("/kaggle/input/newdata/data_test.csv", index_col=None)


In [ ]:
filled_df = filled_df.iloc[:, 1:]
data_test = data_test.iloc[:, 1:]
filled_df = filled_df.iloc[:, 1:]

In [ ]:
#filled_df.info()
#data_test.info()

In [ ]:
#xgboost分類器
import xgboost as xgb
import pandas as pd
import cudf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score
from imblearn.over_sampling import ADASYN
from collections import Counter

# 假設 'filled_df' 是包含數據的 pandas DataFrame
X = filled_df.drop(columns=['y'])  # 移除目標變數作為特徵
y = filled_df['y']  # 目標變數

ada = ADASYN(sampling_strategy=1)
X_train, y_train = ada.fit_resample(X, y)
print('Resampled dataset shape %s' % Counter(y_train))

# 訓練 XGBoost 模型
#model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss")
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

# 取得特徵重要性
feature_importance = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

Resampled dataset shape Counter({1: 193647, 0: 193234})


In [ ]:
# 取得前 80 個重要特徵
selected_features = feature_importance.iloc[:80]['Feature'].tolist()

# 過濾數據集
filled_df = filled_df[selected_features + ['y']]

data_test =data_test[selected_features]



In [ ]:
#xgboost分類器，測試前40-80個特徵
#最好的是0.8228
import xgboost as xgb
import pandas as pd
import cudf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score
from imblearn.over_sampling import ADASYN
from collections import Counter



# 假設 'filled_df' 是包含數據的 pandas DataFrame
X = filled_df.drop(columns=['y'])  # 移除目標變數作為特徵
y = filled_df['y']  # 目標變數

# 切分數據集，80% 用於訓練，20% 用於測試
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ada = ADASYN(sampling_strategy=1, n_neighbors=11)
X_train, y_train = ada.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_train))

# 轉換為 cudf DataFrame（GPU 運算）
X_train_cudf = cudf.DataFrame(X_train)
X_test_cudf = cudf.DataFrame(X_test)
y_train_cudf = y_train.to_numpy()  # 轉為 NumPy，XGBoost 需要 NumPy 格式的標籤
y_test_cudf = y_test.to_numpy()

# 設置 XGBoost 參數（使用 GPU + 設定權重）
params = {
    'objective': 'binary:logistic',  # 二元分類
    'eval_metric': 'logloss',        # 交叉熵損失
    'tree_method': 'hist',       # GPU 加速
    'device':"cuda",                     # 指定使用的 GPU
    'random_state': 42,
    'n_estimators': 10000
}

# 定義一個列表，將要測試的特徵數量
features_list =  list(range(50, 71, 6))


# 迴圈遍歷不同的特徵數量
for features_count in features_list:
    # 選擇前 'features_count' 個特徵
    X_train_subset = X_train.iloc[:, :features_count]
    X_test_subset = X_test.iloc[:, :features_count]
    test_subset =data_test.iloc[:, :features_count]


    # 轉換為 cudf DataFrame
    X_train_cudf = cudf.DataFrame(X_train_subset)
    X_test_cudf = cudf.DataFrame(X_test_subset)
    test_cudf = cudf.DataFrame(test_subset)

    # 初始化 XGBoost 分類器
    xgb_clf = xgb.XGBClassifier(**params)

    # 訓練模型
    xgb_clf.fit(X_train_cudf, y_train_cudf)

    # 預測
    y_pred = xgb_clf.predict(X_test_cudf)

    f1 = f1_score(y_test_cudf, y_pred)
    test_pred = xgb_clf.predict(test_cudf)
    # 顯示 F1 分數和使用的特徵數量
    print(f"使用前 {features_count} 個特徵的 XGBoost GPU 加速的 F1 分數: {f1:.4f}")
    print(f"預測結果的平均值: {np.mean(y_pred)}")
    print(f"測試結果的平均值: {np.mean(test_pred)}")



Resampled dataset shape Counter({1: 154627, 0: 154572})


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()


使用前 50 個特徵的 XGBoost GPU 加速的 F1 分數: 0.8158
預測結果的平均值: 0.005829481253210067
測試結果的平均值: 0.005217460570336148


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()


使用前 56 個特徵的 XGBoost GPU 加速的 F1 分數: 0.8094
預測結果的平均值: 0.005932203389830509
測試結果的平均值: 0.005257288513621157


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()


使用前 62 個特徵的 XGBoost GPU 加速的 F1 分數: 0.8151
預測結果的平均值: 0.005778120184899846
測試結果的平均值: 0.005297116456906165


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()


使用前 68 個特徵的 XGBoost GPU 加速的 F1 分數: 0.8087
預測結果的平均值: 0.005880842321520288
測試結果的平均值: 0.0053767723434761825


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()


In [ ]:
# 取五次平均
import xgboost as xgb
import pandas as pd
import cudf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from imblearn.over_sampling import ADASYN
from collections import Counter

# 假設 'filled_df' 是包含數據的 pandas DataFrame
X = filled_df.drop(columns=['y'])  # 移除目標變數作為特徵
y = filled_df['y']  # 目標變數

# 要測試的特徵數量清單
features_list = list(range(50, 71, 5))

# XGBoost 參數設置
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42,
    'n_estimators': 10000
}

# 主迴圈：針對每一個特徵數量
for features_count in features_list:
    f1_scores = []
    pred_means = []
    test_pred_means = []

    for repeat in range(5):
        # 切分資料集
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42 + repeat
        )

        # ADASYN 過採樣
        ada = ADASYN(sampling_strategy=1, n_neighbors=11)
        X_train, y_train = ada.fit_resample(X_train, y_train)

        # 只取前 N 個特徵
        X_train_subset = X_train.iloc[:, :features_count]
        X_test_subset = X_test.iloc[:, :features_count]
        test_subset = data_test.iloc[:, :features_count]

        # 轉換為 cudf
        X_train_cudf = cudf.DataFrame(X_train_subset)
        X_test_cudf = cudf.DataFrame(X_test_subset)
        test_cudf = cudf.DataFrame(test_subset)

        y_train_cudf = y_train.to_numpy()
        y_test_cudf = y_test.to_numpy()

        # 訓練模型
        xgb_clf = xgb.XGBClassifier(**params)
        xgb_clf.fit(X_train_cudf, y_train_cudf)

        # 預測與計算 F1
        y_pred = xgb_clf.predict(X_test_cudf)
        test_pred = xgb_clf.predict(test_cudf)

        f1 = f1_score(y_test_cudf, y_pred)
        f1_scores.append(f1)
        pred_means.append(np.mean(y_pred))
        test_pred_means.append(np.mean(test_pred))

    # 計算平均值
    avg_f1 = np.mean(f1_scores)
    avg_pred_mean = np.mean(pred_means)
    avg_test_pred_mean = np.mean(test_pred_means)

    print(f"使用前 {features_count} 個特徵的 XGBoost GPU 加速的平均 F1 分數: {avg_f1:.4f}")
    print(f"預測結果的平均值: {avg_pred_mean:.4f}")
    print(f"測試結果的平均值: {avg_test_pred_mean:.4f}")


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

使用前 50 個特徵的 XGBoost GPU 加速的平均 F1 分數: 0.8002
預測結果的平均值: 0.0058
測試結果的平均值: 0.0054


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

使用前 55 個特徵的 XGBoost GPU 加速的平均 F1 分數: 0.7981
預測結果的平均值: 0.0058
測試結果的平均值: 0.0055


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

使用前 60 個特徵的 XGBoost GPU 加速的平均 F1 分數: 0.7997
預測結果的平均值: 0.0058
測試結果的平均值: 0.0054


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

使用前 65 個特徵的 XGBoost GPU 加速的平均 F1 分數: 0.8008
預測結果的平均值: 0.0057
測試結果的平均值: 0.0054


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

使用前 70 個特徵的 XGBoost GPU 加速的平均 F1 分數: 0.8025
預測結果的平均值: 0.0057
測試結果的平均值: 0.0054


In [ ]:
嘗試不同的 KK
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from imblearn.over_sampling import ADASYN
import numpy as np
import cudf
import xgboost as xgb
from collections import Counter

K_list = [3, 5, 7, 9, 11]

# 固定使用前 60 個特徵
feature_count = 70

for K in K_list:
    print(f"\n=== 使用 ADASYN 的 K = {K} ===")

    f1_scores = []
    pred_means = []
    test_pred_means = []

    for repeat in range(5):
        # 切分資料
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42 + repeat
        )

        # ADASYN 過採樣
        ada = ADASYN(sampling_strategy=1, n_neighbors=K)
        X_train, y_train = ada.fit_resample(X_train, y_train)

        # 限定前 N 個特徵
        X_train_subset = X_train.iloc[:, :feature_count]
        X_test_subset = X_test.iloc[:, :feature_count]
        test_subset = data_test.iloc[:, :feature_count]

        # 轉成 cudf 格式
        X_train_cudf = cudf.DataFrame(X_train_subset)
        X_test_cudf = cudf.DataFrame(X_test_subset)
        test_cudf = cudf.DataFrame(test_subset)

        y_train_cudf = y_train.to_numpy()
        y_test_cudf = y_test.to_numpy()

        # 建立模型
        xgb_clf = xgb.XGBClassifier(**params)
        xgb_clf.fit(X_train_cudf, y_train_cudf)

        # 預測
        y_pred = xgb_clf.predict(X_test_cudf)
        test_pred = xgb_clf.predict(test_cudf)

        # 計算 F1 分數
        f1 = f1_score(y_test_cudf, y_pred)
        f1_scores.append(f1)
        pred_means.append(np.mean(y_pred))
        test_pred_means.append(np.mean(test_pred))

    # 輸出結果
    avg_f1 = np.mean(f1_scores)
    avg_pred_mean = np.mean(pred_means)
    avg_test_pred_mean = np.mean(test_pred_means)

    print(f"平均 F1 分數: {avg_f1:.4f}")
    print(f"預測結果的平均值: {avg_pred_mean:.4f}")
    print(f"測試資料的預測平均值: {avg_test_pred_mean:.4f}")



=== 使用 ADASYN 的 K = 3 ===


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

平均 F1 分數: 0.7957
預測結果的平均值: 0.0056
測試資料的預測平均值: 0.0052

=== 使用 ADASYN 的 K = 5 ===


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

平均 F1 分數: 0.7973
預測結果的平均值: 0.0058
測試資料的預測平均值: 0.0054

=== 使用 ADASYN 的 K = 7 ===


...

平均 F1 分數: 0.7970
預測結果的平均值: 0.0058
測試資料的預測平均值: 0.0054

=== 使用 ADASYN 的 K = 9 ===


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

平均 F1 分數: 0.8037
預測結果的平均值: 0.0058
測試資料的預測平均值: 0.0054

=== 使用 ADASYN 的 K = 11 ===


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr

平均 F1 分數: 0.8042
預測結果的平均值: 0.0058
測試資料的預測平均值: 0.0055


In [ ]:
#最終的做法

import xgboost as xgb
import pandas as pd
import cudf
from sklearn.metrics import f1_score
from imblearn.over_sampling import ADASYN
from collections import Counter
import numpy as np

# 假設 'filled_df' 是包含數據的 pandas DataFrame
X = filled_df.drop(columns=['y'])  # 移除目標變數作為特徵
y = filled_df['y']  # 目標變數

# 使用 ADASYN 進行過採樣
ada = ADASYN(sampling_strategy=1, n_neighbors= 11)
X_resampled, y_resampled = ada.fit_resample(X, y)
print('Resampled dataset shape %s' % Counter(y_resampled))

# 轉換為 cudf DataFrame（GPU 運算）
X_resampled_cudf = cudf.DataFrame(X_resampled)
y_resampled_cudf = y_resampled.to_numpy()  # 轉為 NumPy，XGBoost 需要 NumPy 格式的標籤

# 設置 XGBoost 參數（使用 GPU + 設定權重）
params = {
    'objective': 'binary:logistic',  # 二元分類
    'eval_metric': 'logloss',        # 交叉熵損失
    'tree_method': 'hist',           # GPU 加速
    'device': "cuda",                # 指定使用的 GPU
    'random_state': 42,
    'n_estimators': 10000
}

# 設定 features_count = 40
features_count = 70

# 選擇前 40 個特徵
X_resampled_subset = X_resampled.iloc[:, :features_count]

# 轉換為 cudf DataFrame
X_resampled_cudf = cudf.DataFrame(X_resampled_subset)

# 初始化 XGBoost 分類器
xgb_clf = xgb.XGBClassifier(**params)

# 訓練模型
xgb_clf.fit(X_resampled_cudf, y_resampled_cudf)

# 預測
y_pred = xgb_clf.predict(X_resampled_cudf)

# 計算 F1 分數
f1 = f1_score(y_resampled_cudf, y_pred)

# 顯示 F1 分數和使用的特徵數量
print(f"使用前 {features_count} 個特徵的 XGBoost GPU 加速的 F1 分數: {f1:.4f}")
print(f"預測結果的平均值: {np.mean(y_pred)}")


Resampled dataset shape Counter({0: 193234, 1: 193023})


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()
/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()


使用前 70 個特徵的 XGBoost GPU 加速的 F1 分數: 1.0000
預測結果的平均值: 0.499726865791429


In [ ]:
# 假設 filled_test_data 已經處理好，並且模型 xgb_clf 已經訓練好

import cudf
import pandas as pd

data_test_cudf = cudf.DataFrame(data_test.iloc[:, :70])

y_pred = xgb_clf.predict(data_test_cudf)

# 讀取輸出模板 CSV
output_data = pd.read_csv("/kaggle/input/output-data/predictions.csv")

# 確保 output_data 有足夠的列
if output_data.shape[1] > 1:
    output_data.iloc[:, 1] = y_pred
else:
    output_data['prediction'] = y_pred # 如果沒有第二列，就新增一個

# 指定保存的路徑
save_path = "/kaggle/working/xgb_y_pred.csv"

# 保存數據到 CSV
output_data.to_csv(save_path, index=False)
print(f"文件已保存到: {save_path}")
print(f"預測結果的平均值: {np.mean(y_pred)}")


/usr/local/lib/python3.10/dist-packages/xgboost/data.py:839: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  feature_names = data.columns.format()


文件已保存到: /kaggle/working/xgb_y_pred.csv
預測結果的平均值: 0.005217460570336148


In [ ]:
#保存模型

import joblib

# 假設你已經有一個訓練好的模型，例如 xgb_clf
# 保存模型
joblib.dump(xgb_clf, 'xgboost_model(全村的希望).pkl')

# 載入模型
loaded_model = joblib.load('xgboost_model.pkl')

# 使用載入的模型進行預測
predictions = loaded_model.predict(X_test)


In [ ]:
#以下是其他亂嘗試的東西

In [ ]:
df = pd.read_csv("/kaggle/input/new-v1/data_selected.csv", index_col=None)
data_test = pd.read_csv("/kaggle/input/new-v1/data_test.csv", index_col=None)


In [ ]:
df = df.iloc[:, 1:]
data_test = data_test.iloc[:, 1:]

In [ ]:
df_filled.info()
data_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200864 entries, 0 to 200863
Columns: 151 entries, 技術指標_月RSI.10. to y
dtypes: float64(150), int64(1)
memory usage: 231.4 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25108 entries, 0 to 25107
Columns: 150 entries, 技術指標_月RSI.10. to 買超第9名分點前13天賣張
dtypes: float64(150)
memory usage: 28.7 MB


In [ ]:
#可以補缺值的方法
import pandas as pd
import lightgbm as lgb
import numpy as np
from tqdm import tqdm  # 進度條


def fill_missing_with_lightgbm(df, batch_size=10000):
    """
    使用 LightGBM 填補 DataFrame 中的缺失值，並顯示進度條。
    """
    df_filled = df.copy()

    for column in tqdm(df.columns, desc="填補缺失值進度"):  # 加入進度條
        # 找到缺失值的索引
        missing_index = df[df[column].isnull()].index
        non_missing_index = df[df[column].notnull()].index

        if len(missing_index) == 0:
            continue  # 如果某列沒有缺失值，則跳過

        # 準備訓練數據
        X_train = df.drop(columns=[column]).iloc[non_missing_index]
        y_train = df[column].iloc[non_missing_index]

        # 設置 LightGBM 參數（使用 GPU 加速）
        params = {
            'objective': 'regression',  # 回歸任務
            'metric': 'rmse',           # 均方根誤差
            'boosting_type': 'gbdt',    # 梯度提升樹
            'learning_rate': 0.05,      # 學習率
            'num_leaves': 31,           # 最大葉子數
            'verbose': -1,              # 不顯示訓練過程
            'num_threads': 8,           # 使用 8 個 CPU 執行緒
            'device': 'gpu'             # 使用 GPU 加速（如果有 GPU）
        }

        # 訓練 LightGBM
        train_data = lgb.Dataset(X_train, label=y_train)
        model = lgb.train(params, train_data, num_boost_round=100)

        # 預測缺失值
        X_missing = df.drop(columns=[column]).iloc[missing_index]
        predicted_values = model.predict(X_missing)

        # 確保數據類型一致，避免 FutureWarning
        df_filled.loc[missing_index, column] = predicted_values.astype(df[column].dtype)

    return df_filled

# 測試填補
data_test= fill_missing_with_lightgbm(data_test)
#imputed_test = fill_missing_with_lightgbm(data_test.iloc[:,0:50])

填補缺失值進度: 100%|██████████| 150/150 [06:18<00:00,  2.53s/it]


In [ ]:
#憶起補缺值再分開
import pandas as pd
from sklearn.impute import SimpleImputer

combined_df = pd.concat([df.drop(columns=['飆股']), data_test], ignore_index=True)

imputed_data = fill_missing_with_lightgbm(combined_df)
# 使用均值填補缺失值 (可以換成其他方法)



填補缺失值進度: 100%|██████████| 150/150 [14:31<00:00,  5.81s/it]


NameError: name 'train_df' is not defined

In [ ]:
# 重新拆分回原來的部分
df_filled = imputed_data.iloc[:len(df), :]  # 取回原來的 train
test_data = imputed_data.iloc[len(df):, :]  # 取回原來的 test

In [ ]:
len(df_filled)

200864

In [ ]:
df["飆股"]

0         0
1         0
2         0
3         0
4         0
         ..
200859    0
200860    0
200861    0
200862    0
200863    0
Name: 飆股, Length: 200864, dtype: int64

In [ ]:
df_filled.loc[:, "y"] = df["飆股"]

In [ ]:

#我跑不動 CNN的方法
import pandas as pd
import numpy as np
import cudf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import ADASYN

# 假設 'filled_df' 是包含數據的 pandas DataFrame
X = filled_df.drop(columns=['y'])  # 移除目標變數作為特徵
y = filled_df['y']  # 目標變數

# 切分數據集，80% 用於訓練，20% 用於測試
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.iloc[:, :40]
X_test = X_test.iloc[:, 40]
test_subset = data_test.iloc[:, :40]

# 使用 ADASYN 進行過採樣
ada = ADASYN()
X_train_resampled, y_train_resampled = ada.fit_resample(X_train, y_train)

# 轉換為 cudf DataFrame（GPU 運算）
X_train_cudf = cudf.DataFrame(X_train_resampled)
X_test_cudf = cudf.DataFrame(X_test)
test_cudf = cudf.DataFrame(test_subset)

# 轉換為 NumPy 陣列
X_train_np = X_train_cudf.to_pandas().values  # 轉回 pandas，然後轉換為 NumPy 陣列
X_test_np = X_test_cudf.to_pandas().values
test_np = test_cudf.to_pandas().values

# 將數據重塑為 (samples, timesteps, features) 格式，這是 CNN 所需的格式
X_train_np = X_train_np.reshape(X_train_np.shape[0], X_train_np.shape[1], 1)
X_test_np = X_test_np.reshape(X_test_np.shape[0], X_test_np.shape[1], 1)
test_np = test_np.reshape(test_np.shape[0], test_np.shape[1], 1)

# 構建 1D CNN 模型
model = Sequential()

# 卷積層
model.add(Conv1D(filters=128, kernel_size=3, activation='relu', input_shape=(X_train_np.shape[1], 1)))

# 可以選擇使用池化層，或完全去掉
# 如果使用池化層，請使用更大的 kernel_size 和 pool_size
model.add(MaxPooling1D(pool_size=2))  # 試著使用 larger pool_size

# 添加 Dropout 層來防止過擬合
model.add(Dropout(0.5))

# Flatten 層，將多維數據展平
model.add(Flatten())

# 全連接層
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))  # 防止過擬合

# 輸出層
model.add(Dense(1, activation='sigmoid'))

# 編譯模型
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# 訓練模型
history = model.fit(X_train_np, y_train_resampled, epochs=5, batch_size=32, validation_data=(X_test_np, y_test))

# 預測
y_pred_cnn = model.predict(X_test_np)
pred_cnn = model.predict(test_np)

# 預測結果轉換為 0 或 1
y_pred_cnn = (y_pred_cnn > 0.5).astype(int)
pred_cnn = (pred_cnn > 0.5).astype(int)

# 計算準確率
from sklearn.metrics import f1_score

# 計算預測結果的平均值
y_pred_mean = np.mean(y_pred_cnn)
pred_mean = np.mean(pred_cnn)



print(f"預測結果的平均值: {y_pred_mean:.4f}")
print(f"測試結果的平均值: {pred_mean:.4f}")

# 計算 F1 分數
f1 = f1_score(y_test, y_pred_cnn)
print(f"CNN 的 F1 分數: {f1:.4f}")


In [ ]:
# BalancedRandomForestClassifier效果不好
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import f1_score
from imblearn.over_sampling import ADASYN
from collections import Counter
import cudf
import pandas as pd
from sklearn.model_selection import train_test_split

# 假設 'filled_df' 是包含數據的 pandas DataFrame
X = filled_df.drop(columns=['飆股'])  # 移除目標變數作為特徵
y = filled_df['飆股']  # 目標變數

# 切分數據集，80% 用於訓練，20% 用於測試
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 定義特徵數量列表
features_list = [50, 100, 150, 200]

# 迴圈遍歷不同的特徵數量
for features_count in features_list:
    # 選擇前 'features_count' 個特徵
    X_train_subset = X_train.iloc[:, :features_count]
    X_test_subset = X_test.iloc[:, :features_count]

    # 轉換為 cudf DataFrame（GPU 運算）
    X_train_cudf = cudf.DataFrame(X_train_subset)
    X_test_cudf = cudf.DataFrame(X_test_subset)
    y_train_cudf = cudf.Series(y_train.values)  # 轉換目標變數為 cudf Series

    # 初始化 BalancedRandomForestClassifier
    rf_clf = BalancedRandomForestClassifier(n_estimators=1000, random_state=42)

    # 訓練模型
    rf_clf.fit(X_train_subset, y_train)  # 使用 CPU 訓練

    # 使用測試集進行預測
    y_pred = rf_clf.predict(X_test_subset)

    # 計算 F1 分數
    f1 = f1_score(y_test, y_pred)

    # 顯示 F1 分數和使用的特徵數量
    print(f"使用前 {features_count} 個特徵的 BalancedRandomForestClassifier F1 分數: {f1:.4f}")


In [ ]:
#補缺值的方法

import pandas as pd
import lightgbm as lgb
import numpy as np
from tqdm import tqdm  # 引入 tqdm 來顯示進度條
# 取前 200 個特徵（包含 '飆股'）
df = df.iloc[:, :100].copy()
if '飆股' in df.columns:
    df['飆股'] = df['飆股']  # 保留 '飆股' 欄位

# 取前 200 個特徵
data_test = data_test.iloc[:, :100].copy()


def fill_missing_with_lightgbm(df, data_test):
    df_temp = df.drop(columns=['飆股'], errors='ignore')  # 忽略 '飆股' 欄位
    df_temp['source'] = 'train'  # 標記數據來源
    data_test['source'] = 'test'

    combined = pd.concat([df_temp, data_test], axis=0)  # 合併數據集

    # 移除 source 標記
    combined_no_source = combined.drop(columns=['source'])

    # 使用 tqdm 來顯示進度
    for col in tqdm(combined_no_source.columns, desc="填補缺失值進度"):
        if combined_no_source[col].isnull().sum() > 0:  # 只補有缺失值的欄位
            known = combined_no_source[combined_no_source[col].notnull()]
            unknown = combined_no_source[combined_no_source[col].isnull()]

            if known.shape[0] == 0 or unknown.shape[0] == 0:
                continue  # 跳過完全沒有數據或完全沒有缺失的情況

            # 特徵與目標
            X_known = known.drop(columns=[col])
            y_known = known[col]
            X_unknown = unknown.drop(columns=[col])

            # 設定 LightGBM 使用 GPU
            model = lgb.LGBMRegressor(device="gpu") if y_known.dtype.kind in 'ifc' else lgb.LGBMClassifier(device="gpu")
            model.fit(X_known, y_known)

            # 預測填補缺失值
            combined_no_source.loc[combined_no_source[col].isnull(), col] = model.predict(X_unknown)

    # 拆分回原本的 df 和 test_data
    filled_df = combined_no_source.loc[combined['source'] == 'train'].copy()
    filled_test_data = combined_no_source.loc[combined['source'] == 'test'].copy()

    # 加回 '飆股' 欄位
    if '飆股' in df.columns:
        filled_df['飆股'] = df['飆股'].values

    return filled_df, filled_test_data

# 進行補值
filled_df, filled_test_data = fill_missing_with_lightgbm(df, data_test)


In [ ]:
#補缺值後標準化的方法

import pandas as pd
from sklearn.preprocessing import StandardScaler


# 我們需要標準化 'filled_df' 和 'filled_test_data'，去除 '飆股' 欄位
X_train_scaled = filled_df.drop(columns=['飆股'], errors='ignore')  # 去除 '飆股' 欄位
X_test_scaled = filled_test_data

# 設置標準化器
scaler = StandardScaler()

# 對訓練數據進行標準化
X_train_scaled = scaler.fit_transform(X_train_scaled)

# 對測試數據進行標準化（使用訓練數據的均值和標準差）
X_test_scaled = scaler.transform(X_test_scaled)

# 將標準化後的數據轉換為 DataFrame
filled_df_scaled = pd.DataFrame(X_train_scaled, columns=filled_df.drop(columns=['飆股'], errors='ignore').columns)
filled_test_data_scaled = pd.DataFrame(X_test_scaled, columns=filled_test_data.columns)

# 加回 '飆股' 欄位
filled_df_scaled['飆股'] = filled_df['飆股'].values


# 返回標準化後的數據
filled_df_scaled, filled_test_data_scaled
